# 02 - Feature Engineering

This notebook demonstrates the feature extraction pipeline that converts
raw joint angle time series into ML-ready feature vectors.

## Key Features
- Statistical summaries (mean, std, max, min, range) per joint per gait cycle
- Gait phase values (initial contact, midstance, toe-off, peak swing)
- Angular velocity metrics (max, mean)
- Bilateral asymmetry ratios (left vs right)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.compwalk_loader import build_dataset
from src.features.gait_features import extract_cycle_features, compute_asymmetry
from src.features.feature_pipeline import (
    build_feature_matrix, FEATURE_NAMES, N_FEATURES,
    extract_features_from_timeseries,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## Step 1: Load Dataset

In [ ]:
dataset = build_dataset()
print(f"Records: {len(dataset)}, Participants: {dataset['participant_id'].nunique()}")

## Step 2: Feature Extraction Demo

Let's walk through extracting features from a single participant.

In [ ]:
# Take one participant's data
sample_pid = dataset['participant_id'].unique()[0]
sample = dataset[dataset['participant_id'] == sample_pid]
print(f"Participant: {sample_pid}")
print(f"Cohort: {sample['cohort'].iloc[0]}")
print(f"Joints available: {sample['joint'].unique()}")

# Extract cycle features for one joint
knee_data = sample[sample['joint'] == 'knee_flexion']
if not knee_data.empty:
    ts = knee_data.iloc[0]['angle_timeseries']
    features = extract_cycle_features(ts)
    print(f"\nKnee flexion features:")
    for k, v in features.items():
        print(f"  {k}: {v:.2f}")

## Step 3: Build Full Feature Matrix

In [ ]:
X, y, pids = build_feature_matrix(dataset)
print(f"Feature matrix: {X.shape}")
print(f"Class distribution: healthy={sum(y==0)}, injured={sum(y==1)}")
print(f"Total features: {N_FEATURES}")

## Step 4: Feature Analysis

In [ ]:
# Top discriminative features (by t-statistic)
healthy = X[y == 0]
injured = X[y == 1]

separations = []
for i in range(X.shape[1]):
    h_std = np.std(healthy[:, i]) + 1e-8
    i_std = np.std(injured[:, i]) + 1e-8
    pooled = np.sqrt((h_std**2 + i_std**2) / 2)
    t = abs(np.mean(healthy[:, i]) - np.mean(injured[:, i])) / pooled
    separations.append(t)

top_idx = np.argsort(separations)[-10:][::-1]
print("Top 10 most discriminative features:")
for i, idx in enumerate(top_idx):
    print(f"  {i+1}. {FEATURE_NAMES[idx]}: t={separations[idx]:.3f}")

In [ ]:
# Violin plots for top features
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax_idx, feat_idx in enumerate(top_idx[:8]):
    data = pd.DataFrame({
        'Value': np.concatenate([healthy[:, feat_idx], injured[:, feat_idx]]),
        'Group': ['Healthy'] * len(healthy) + ['ACL Injured'] * len(injured),
    })
    sns.violinplot(data=data, x='Group', y='Value', ax=axes[ax_idx],
                   palette=['#4CAF50', '#F44336'])
    axes[ax_idx].set_title(FEATURE_NAMES[feat_idx].replace('_', ' ').title()[:30], fontsize=9)
    axes[ax_idx].set_xlabel('')

fig.suptitle('Feature Distributions: Healthy vs ACL Injured', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation matrix
from src.visualization.plots import plot_correlation_matrix
plot_correlation_matrix(X, FEATURE_NAMES)
plt.show()